# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
from io import StringIO
from IPython.display import display

BESTAND = Path("Datasheet_query_(10).csv")
if not BESTAND.exists():
    raise FileNotFoundError(f"CSV niet gevonden: {BESTAND.resolve()}")

with BESTAND.open("r", encoding="utf-8-sig") as f:
    regels = f.readlines()

tabellen, huidige = [], []
for regel in regels:
    if regel.startswith("#datatype"):
        if huidige:
            tabellen.append(huidige)
        huidige = []
    elif regel.startswith("#") or not regel.strip():
        continue
    else:
        huidige.append(regel)
if huidige:
    tabellen.append(huidige)

frames = []
for tabel in tabellen:
    frame = pd.read_csv(StringIO("".join(tabel)), sep=",")
    frame = frame.loc[:, ~frame.columns.str.startswith("Unnamed")]
    frame = frame.drop(columns=["result", "table", "_start", "_stop"], errors="ignore")
    frame = frame.dropna(how="all")
    if "_time" in frame.columns:
        frame["_time"] = pd.to_datetime(frame["_time"], utc=True, errors="coerce")
    frames.append(frame)

solar_frames = [f for f in frames if "impp" in f.columns]
weather_frames = [f for f in frames if "Temperature-PVMET500" in f.columns]
if not solar_frames or not weather_frames:
    raise ValueError("Solar- of weerdata niet gevonden. Controleer de CSV-structuur.")

df_solar_raw = pd.concat(solar_frames, ignore_index=True)
df_weather = pd.concat(weather_frames, ignore_index=True)
print("Zonnepaneelrijen:", len(df_solar_raw))
print("Weerrijen:", len(df_weather))

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:
df_solar = df_solar_raw.copy()
pt_cols = [c for c in df_solar.columns if c.startswith("Pt100") and c != "Pt100"]
if pt_cols:
    pt = df_solar[pt_cols].apply(pd.to_numeric, errors="coerce").bfill(axis=1).iloc[:, 0]
    if "Pt100" in df_solar.columns:
        df_solar["Pt100"] = pd.to_numeric(df_solar["Pt100"], errors="coerce").combine_first(pt)
    else:
        df_solar["Pt100"] = pt
    df_solar = df_solar.drop(columns=pt_cols)

variabelen = ["Pt100", "ff", "impp", "isc", "vmpp", "voc", "CMP-10", "Ref"]
for col in variabelen:
    if col in df_solar.columns:
        df_solar[col] = pd.to_numeric(df_solar[col], errors="coerce")
for col in ["Temperature-PVMET500", "Humidity-PVMET500", "Pressure-PVMET500"]:
    df_weather[col] = pd.to_numeric(df_weather[col], errors="coerce")

df_solar = df_solar.dropna(subset=["_time", "label"]).sort_values(["label", "_time"]).copy()
df_weather = df_weather.dropna(subset=["_time"]).sort_values("_time").copy()

display(df_solar.groupby("label")[["Pt100", "ff", "impp", "isc", "vmpp", "voc"]].count())
print("Beschikbare weersensorwaarden:")
display(df_weather[["Temperature-PVMET500", "Humidity-PVMET500", "Pressure-PVMET500"]].count())

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:
# Corrigeer alleen omgevingstemperaturen boven 40 °C.
# Werk op een chronologisch gesorteerde kopie; het origineel blijft beschikbaar.
df_weather = df_weather.sort_values("_time").reset_index(drop=True).copy()
kolom = "Temperature-PVMET500"
df_weather[f"{kolom}_raw"] = df_weather[kolom].copy()

origineel = df_weather[f"{kolom}_raw"]
fout = origineel.gt(40) & origineel.notna()
geldig = origineel.notna() & ~fout

# Bepaal voor ieder punt de dichtstbijzijnde geldige buur aan beide kanten.
vorige_geldig = origineel.where(geldig).ffill().shift(1)
volgende_geldig = origineel.where(geldig).bfill().shift(-1)
gecorrigeerd = (vorige_geldig + volgende_geldig) / 2

df_weather.loc[fout, kolom] = gecorrigeerd.loc[fout]

correcties = df_weather.loc[fout, ["_time", f"{kolom}_raw", kolom]].copy()
correcties.columns = ["Tijdstip (UTC)", "Oorspronkelijk (°C)", "Gecorrigeerd (°C)"]
correcties["Correctie mogelijk"] = correcties["Gecorrigeerd (°C)"].notna()

print(f"Aantal omgevingstemperaturen boven 40 °C: {fout.sum()}")
print(f"Succesvol gecorrigeerd: {correcties['Correctie mogelijk'].sum()}")
print(f"Niet te corrigeren (ontbrekende buur): {(~correcties['Correctie mogelijk']).sum()}")
display(correcties)

# Interactieve kwaliteitscontrole: vóór en na de correctie
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_weather["_time"], y=df_weather[f"{kolom}_raw"],
    mode="lines+markers", name="Originele PVMET500",
    line=dict(color="#d06b59", width=1), marker=dict(size=3),
    hovertemplate="Tijd: %{x|%d-%m-%Y %H:%M} UTC<br>Origineel: %{y:.2f} °C<extra></extra>"
))
fig.add_trace(go.Scatter(
    x=df_weather["_time"], y=df_weather[kolom],
    mode="lines", name="Gecorrigeerde PVMET500",
    line=dict(color="#137c72", width=2),
    hovertemplate="Tijd: %{x|%d-%m-%Y %H:%M} UTC<br>Gecorrigeerd: %{y:.2f} °C<extra></extra>"
))
fig.update_layout(
    title="2.1. Omgevingstemperatuur vóór en na correctie",
    xaxis_title="Datum en tijd (UTC)",
    yaxis_title="Omgevingstemperatuur PVMET500 (°C)",
    template="plotly_white", height=510)
fig.show()

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:
weer = (df_weather[["_time", "Temperature-PVMET500", "Humidity-PVMET500", "Pressure-PVMET500"]]
        .drop_duplicates(subset=["_time"])
        .sort_values("_time"))
solar_sorted = df_solar.sort_values("_time").copy()
df = pd.merge_asof(
    solar_sorted, weer, on="_time",
    direction="nearest", tolerance=pd.Timedelta("2min"),
    suffixes=("", "_weer")
)
df["delta_T"] = df["Pt100"] - df["Temperature-PVMET500"]
df["P_mpp"] = df["impp"] * df["vmpp"]
print(f"Gekoppelde rijen: {len(df):,}")
print(f"Met geldige omgevingstemperatuur: {df['Temperature-PVMET500'].notna().sum():,}")
display(df.groupby("label")[["Pt100", "delta_T", "P_mpp"]].count())

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:
# Figuur 4a: interactieve tijdreeksen
fig = go.Figure()
for paneel, g in sorted(df.groupby("label")):
    g = g.dropna(subset=["_time", "Pt100"]).sort_values("_time")
    fig.add_trace(go.Scatter(
        x=g["_time"], y=g["Pt100"], mode="lines+markers",
        name=paneel.replace("PvDevice-", "PV"),
        marker=dict(size=4), line=dict(width=1.5),
        hovertemplate="Tijd: %{x|%d-%m-%Y %H:%M} UTC<br>Paneeltemperatuur: %{y:.2f} °C<extra>%{fullData.name}</extra>"
    ))
fig.add_trace(go.Scatter(
    x=weer["_time"], y=weer["Temperature-PVMET500"],
    mode="lines", name="Omgeving (PVMET500)",
    line=dict(color="#1d2430", width=2.5),
    hovertemplate="Tijd: %{x|%d-%m-%Y %H:%M} UTC<br>Omgevingstemperatuur: %{y:.2f} °C<extra></extra>"
))
fig.update_layout(title="4a. Paneeltemperatuur en omgevingstemperatuur",
    xaxis_title="Datum en tijd (UTC)", yaxis_title="Temperatuur (°C)",
    template="plotly_white", hovermode="x unified", height=530,
    legend_title_text="Sensor / paneel")
fig.show()

# Figuur 4b: temperatuurverschil als boxplot
fig = go.Figure()
for paneel, g in sorted(df.groupby("label")):
    g = g.dropna(subset=["delta_T"])
    fig.add_trace(go.Box(
        y=g["delta_T"], name=paneel.replace("PvDevice-", "PV"),
        boxpoints="outliers",
        hovertemplate="ΔT: %{y:.2f} °C<extra>%{fullData.name}</extra>"
    ))
fig.add_hline(y=0, line_dash="dash", line_color="gray",
              annotation_text="Paneel = omgeving")
fig.update_layout(title="4b. Verschil tussen paneel- en omgevingstemperatuur",
    xaxis_title="Zonnepaneel", yaxis_title="ΔT = T_paneel − T_omgeving (°C)",
    template="plotly_white", height=500, showlegend=False)
fig.show()

display(df.groupby("label")["delta_T"].agg(["count", "median", "min", "max"]).round(2))

# Figuur 4c: delta T van elk paneel als functie van de tijd
fig = go.Figure()

for paneel, groep in sorted(df.groupby("label")):
    g = groep.dropna(subset=["_time", "delta_T"]).sort_values("_time").copy()
    if g.empty:
        continue

    # Onderbreek lijnen bij ontbrekende metingen of gaten > 20 minuten.
    tijdsgat = g["_time"].diff() > pd.Timedelta(minutes=20)
    x, y, tijdstippen = [], [], []
    for tijd, verschil, gat in zip(g["_time"], g["delta_T"], tijdsgat):
        if gat:
            x.append(None)
            y.append(None)
            tijdstippen.append(None)
        x.append(tijd)
        y.append(verschil)
        tijdstippen.append(tijd.strftime("%d-%m-%Y %H:%M UTC"))

    fig.add_trace(go.Scatter(
        x=x, y=y, customdata=tijdstippen,
        mode="lines+markers",
        name=paneel.replace("PvDevice-", "PV"),
        line=dict(width=1.8),
        marker=dict(size=5),
        connectgaps=False,
        hovertemplate=(
            "Tijd: %{customdata}<br>"
            "ΔT (paneel − omgeving): %{y:.2f} °C"
            "<extra>%{fullData.name}</extra>"
        )
    ))

fig.add_hline(
    y=0, line_dash="dash", line_color="gray",
    annotation_text="ΔT = 0 °C"
)
fig.update_layout(
    title="4c. Temperatuurverschil tussen zonnepanelen en omgeving door de tijd",
    xaxis_title="Datum en tijd (UTC)",
    yaxis_title="ΔT = T_paneel − T_omgeving (°C)",
    template="plotly_white",
    height=560,
    hovermode="closest",
    legend_title_text="Zonnepaneel"
)
fig.show()

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:
# Drie afzonderlijke interactieve scatterplots
grafieken = [
    ("voc", "5a. Paneeltemperatuur versus openklemspanning", "Openklemspanning Voc (V)", "V"),
    ("ff", "5b. Paneeltemperatuur versus fill factor", "Fill factor FF (vermoedelijk %)", "%"),
    ("P_mpp", "5c. Paneeltemperatuur versus berekend vermogen", "Berekend maximaal vermogen Pₘₚₚ (W)", "W"),
]
for ykolom, titel, yas, eenheid in grafieken:
    fig = go.Figure()
    for paneel, g in sorted(df.groupby("label")):
        g = g.dropna(subset=["Pt100", ykolom])
        fig.add_trace(go.Scatter(
            x=g["Pt100"], y=g[ykolom], mode="markers",
            name=paneel.replace("PvDevice-", "PV"),
            marker=dict(size=7, opacity=0.65),
            customdata=g["_time"].dt.strftime("%d-%m-%Y %H:%M UTC"),
            hovertemplate=(
                "Tijd: %{customdata}<br>"
                "Paneeltemperatuur: %{x:.2f} °C<br>"
                f"{yas}: " + "%{y:.2f} " + eenheid +
                "<extra>%{fullData.name}</extra>"
            )
        ))
    fig.update_layout(title=titel,
        xaxis_title="Paneeltemperatuur Pt100 (°C)", yaxis_title=yas,
        template="plotly_white", height=520,
        legend_title_text="Zonnepaneel")
    fig.show()

for ykolom, _, _, _ in grafieken:
    print(f"\nPearson-correlatie Pt100 versus {ykolom} (beschrijvend):")
    resultaten = {}
    for paneel, g in df.groupby("label"):
        paar = g[["Pt100", ykolom]].dropna()
        resultaten[paneel] = paar["Pt100"].corr(paar[ykolom]) if len(paar) >= 3 else np.nan
    display(pd.Series(resultaten, name="Pearson r").round(3).to_frame())

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:
if "CMP-10" in df.columns:
    cmp = (df[["_time", "CMP-10"]].dropna()
           .groupby("_time", as_index=False)["CMP-10"].median()
           .sort_values("_time"))
    fig = go.Figure(go.Scatter(
        x=cmp["_time"], y=cmp["CMP-10"], mode="lines+markers",
        name="CMP-10", line=dict(color="#b88122", width=2),
        marker=dict(size=4),
        hovertemplate="Tijd: %{x|%d-%m-%Y %H:%M} UTC<br>CMP-10: %{y:.2f} (eenheid te verifiëren)<extra></extra>"
    ))
    fig.update_layout(
        title="6. CMP-10 — vermoedelijke zoninstraling (nog te verifiëren)",
        xaxis_title="Datum en tijd (UTC)",
        yaxis_title="CMP-10 (vermoedelijk W/m²; niet bevestigd)",
        template="plotly_white", height=520)
    fig.show()
    display(cmp["CMP-10"].describe().round(2))
else:
    print("Geen CMP-10-kolom aanwezig.")

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:
display(df.groupby("label")[["Pt100", "impp", "isc", "vmpp", "voc", "P_mpp"]].count())
stroom = df.pivot_table(index="_time", columns="label", values="impp", aggfunc="first")
records = []
from itertools import combinations
for a, b in combinations(stroom.columns, 2):
    paar = stroom[[a, b]].dropna()
    if len(paar):
        gelijk = np.isclose(paar[a], paar[b], rtol=0, atol=1e-9)
        records.append({"Paneel A": a, "Paneel B": b,
                        "Gelijktijdig": len(paar), "Identiek": int(gelijk.sum()),
                        "Overeenkomst (%)": round(100*gelijk.mean(), 1)})
vergelijking = pd.DataFrame(records)
if not vergelijking.empty:
    display(vergelijking.sort_values("Overeenkomst (%)", ascending=False).head(15))

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:
analyse = df.copy()
analyse["dag_utc"] = analyse["_time"].dt.strftime("%Y-%m-%d")
meetdekking = (analyse.groupby(["label", "dag_utc"])["P_mpp"]
               .count().reset_index(name="aantal"))
fig = px.bar(meetdekking, x="dag_utc", y="aantal", color="label",
             barmode="group", template="plotly_white",
             title="9a. Aantal geldige vermogensmetingen per paneel en dag",
             labels={"dag_utc": "Kalenderdag (UTC)", "aantal": "Aantal Pₘₚₚ-metingen (–)",
                     "label": "Zonnepaneel"})
fig.update_traces(hovertemplate="Dag: %{x}<br>Aantal: %{y} metingen<extra>%{fullData.name}</extra>")
fig.show()
display(analyse.groupby("label")[["Pt100","Temperature-PVMET500","CMP-10","impp","vmpp","voc","ff","P_mpp"]].count())

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:
qc = analyse.copy()
qc["FF_berekend_pct"] = 100 * qc["P_mpp"] / (qc["isc"] * qc["voc"])
qc.loc[(qc["isc"] <= 0) | (qc["voc"] <= 0), "FF_berekend_pct"] = np.nan
qc["FF_verschil_pp"] = qc["ff"] - qc["FF_berekend_pct"]
ff_valid = qc.dropna(subset=["ff","FF_berekend_pct"]).copy()
fig = px.scatter(ff_valid, x="ff", y="FF_berekend_pct", color="label",
                 hover_data=["_time","FF_verschil_pp"], template="plotly_white",
                 title="10a. Gerapporteerde versus berekende fill factor",
                 labels={"ff":"Gerapporteerde FF (exporteenheid; vermoedelijk %)",
                         "FF_berekend_pct":"Berekende FF = 100 × Pₘₚₚ / (Isc × Voc) (%)",
                         "label":"Zonnepaneel"})
if len(ff_valid):
    grenzen = [float(min(ff_valid["ff"].min(),ff_valid["FF_berekend_pct"].min())),
               float(max(ff_valid["ff"].max(),ff_valid["FF_berekend_pct"].max()))]
    fig.add_trace(go.Scatter(x=grenzen,y=grenzen,mode="lines",
                             name="Perfecte overeenkomst",line=dict(dash="dash",color="gray")))
fig.show()
display(qc.groupby("label")["FF_verschil_pp"].agg(["count","median","min","max"]).round(3))
print("Let op: afwijkingen worden eerst onderzocht; geen automatische verwijdering.")

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:
from scipy.stats import linregress

fig = go.Figure()
regressies = []
for paneel, groep in sorted(analyse.groupby("label")):
    paar = groep[["Pt100","voc"]].dropna()
    fig.add_trace(go.Scatter(x=paar["Pt100"],y=paar["voc"],mode="markers",
                             name=str(paneel),marker=dict(size=6,opacity=.55),
                             hovertemplate="Pt100: %{x:.2f} °C<br>Voc: %{y:.2f} V<extra>%{fullData.name}</extra>"))
    if len(paar)>=10 and paar["Pt100"].nunique()>2:
        fit = linregress(paar["Pt100"],paar["voc"])
        xx = np.linspace(paar["Pt100"].min(),paar["Pt100"].max(),60)
        fig.add_trace(go.Scatter(x=xx,y=fit.intercept+fit.slope*xx,mode="lines",
                                 name=f"{paneel} regressie",showlegend=False,
                                 line=dict(dash="dash")))
        regressies.append({"paneel":paneel,"n":len(paar),
                           "helling_V_per_C":fit.slope,"r":fit.rvalue,
                           "p_waarde":fit.pvalue})
fig.update_layout(title="11a. Voc en paneeltemperatuur — ongecorrigeerde samenhang",
                  xaxis_title="Paneeltemperatuur Pt100 (°C)",
                  yaxis_title="Openklemspanning Voc (V)",
                  template="plotly_white",height=580)
fig.show()
regressietabel = pd.DataFrame(regressies)
display(regressietabel.round(4) if not regressietabel.empty else "Onvoldoende gekoppelde metingen.")
print("Waarschuwing: p-waarden veronderstellen onafhankelijke observaties; tijdreeksen zijn doorgaans autocorreleerd.")

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.

In [ ]:
stroom = analyse.pivot_table(index="_time",columns="label",values="impp",aggfunc="first")
panelen = list(stroom.columns)
gelijke_pct = pd.DataFrame(np.nan,index=panelen,columns=panelen)
overlap_n = pd.DataFrame(0,index=panelen,columns=panelen)
for p1 in panelen:
    for p2 in panelen:
        paar = stroom[[p1,p2]].dropna() if p1!=p2 else stroom[[p1]].dropna()
        overlap_n.loc[p1,p2] = len(paar)
        if len(paar)>=5:
            gelijke_pct.loc[p1,p2] = 100.0 if p1==p2 else 100*np.mean(
                np.isclose(paar.iloc[:,0],paar.iloc[:,1],rtol=0,atol=1e-12))
fig = go.Figure(go.Heatmap(x=panelen,y=panelen,z=gelijke_pct.values,
                           customdata=overlap_n.values,
                           zmin=0,zmax=100,colorscale="Blues",
                           colorbar=dict(title="Exact gelijk (%)"),
                           hovertemplate="Paneel X: %{x}<br>Paneel Y: %{y}<br>Exact gelijk: %{z:.1f}%<br>Overlap: %{customdata} metingen<extra></extra>"))
fig.update_layout(title="12a. Overeenkomst van Impp tussen panelen",
                  xaxis_title="Zonnepaneel",yaxis_title="Zonnepaneel",
                  template="plotly_white",height=600)
fig.show()
display(gelijke_pct.round(1))

# SolarNL research workflow

This public copy contains analysis code only. Measurement data, executed outputs, numerical summaries, and interpretations are withheld because the project data are confidential. The private project copy is required to execute this notebook.